In [39]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("ClasePySpark")
    .master("local[8]")                       # 4 núcleos
    .config("spark.driver.memory", "6g")       # 8 GB para el driver
    .config("spark.executor.memory", "4g")     # 4 GB para ejecutores
    .config("spark.sql.shuffle.partitions", "8")  # Particiones para shuffles
    .config("spark.sql.adaptive.enabled", "true") # Ejecución adaptativa (AQE)
    .getOrCreate()
)

In [40]:
# Reducimos logs verbosos
spark.sparkContext.setLogLevel("ERROR")

print(f" Spark versión: {spark.version}")
print(f"   App Name  : {spark.sparkContext.appName}")
print(f"   Master    : {spark.sparkContext.master}")

 Spark versión: 3.5.0
   App Name  : ClasePySpark
   Master    : local[8]


In [41]:
df = spark.read.parquet('data/yellow_tripdata_2010-01.parquet') 

df.printSchema()

root
 |-- vendor_id: string (nullable = true)
 |-- pickup_datetime: string (nullable = true)
 |-- dropoff_datetime: string (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- rate_code: string (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- surcharge: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)



In [42]:
df.show(5)

+---------+-------------------+-------------------+---------------+------------------+------------------+---------------+---------+------------------+------------------+----------------+------------+-----------+---------+-------+----------+------------+------------+
|vendor_id|    pickup_datetime|   dropoff_datetime|passenger_count|     trip_distance|  pickup_longitude|pickup_latitude|rate_code|store_and_fwd_flag| dropoff_longitude|dropoff_latitude|payment_type|fare_amount|surcharge|mta_tax|tip_amount|tolls_amount|total_amount|
+---------+-------------------+-------------------+---------------+------------------+------------------+---------------+---------+------------------+------------------+----------------+------------+-----------+---------+-------+----------+------------+------------+
|      VTS|2010-01-26 07:41:00|2010-01-26 07:45:00|              1|              0.75|        -73.956778|       40.76775|        1|              NULL|        -73.965957|       40.765232|         CAS|

In [43]:
print(f"Filas totales : {df.count():,}")
print(f"   Columnas      : {len(df.columns)}")

Filas totales : 14,863,778
   Columnas      : 18


In [13]:
for col in df.columns :
    print(col)

vendor_id
pickup_datetime
dropoff_datetime
passenger_count
trip_distance
pickup_longitude
pickup_latitude
rate_code
store_and_fwd_flag
dropoff_longitude
dropoff_latitude
payment_type
fare_amount
surcharge
mta_tax
tip_amount
tolls_amount
total_amount


In [22]:
# Estadísticas descriptivas básicas
df.select("trip_distance", 'payment_type', "fare_amount", 'passenger_count', "total_amount") \
  .describe() \
  .show()

+-------+------------------+------------+-----------------+------------------+------------------+
|summary|     trip_distance|payment_type|      fare_amount|   passenger_count|      total_amount|
+-------+------------------+------------+-----------------+------------------+------------------+
|  count|          14863778|    14863778|         14863778|          14863778|          14863778|
|   mean|2.6282668161494915|        NULL|9.427919490578939| 1.684685952656182|11.061816638364652|
| stddev| 3.018446312222399|        NULL|7.379477377412955|1.2606320111578033| 8.594385928232676|
|    min|               0.0|         CAS|              2.5|                 0|               2.5|
|    max|              50.0|         No |            200.0|               208|             236.0|
+-------+------------------+------------+-----------------+------------------+------------------+



In [23]:
df.select("trip_distance", 'payment_type', "fare_amount", 'passenger_count', "total_amount") \
  .summary() \
  .show()

+-------+------------------+------------+-----------------+------------------+------------------+
|summary|     trip_distance|payment_type|      fare_amount|   passenger_count|      total_amount|
+-------+------------------+------------+-----------------+------------------+------------------+
|  count|          14863778|    14863778|         14863778|          14863778|          14863778|
|   mean|2.6282668161494915|        NULL|9.427919490578939| 1.684685952656182|11.061816638364652|
| stddev| 3.018446312222399|        NULL|7.379477377412955|1.2606320111578033| 8.594385928232676|
|    min|               0.0|         CAS|              2.5|                 0|               2.5|
|    25%|               1.0|        NULL|              5.3|                 1|              6.44|
|    50%|              1.68|        NULL|              7.3|                 1| 8.599999999999998|
|    75%|              2.96|        NULL|             10.5|                 2|              12.2|
|    max|           

In [64]:
#Numero de columnas del df
print(f'El dataset tiene {len(df.columns)} columnas')

El dataset tiene 18 columnas


In [68]:
#Tipo de datos en fare_amont
tipo = df.select('fare_amount').dtypes[0][1]
print(f'El tipo de dato en la columna fare_amount es: {tipo}')

El tipo de dato en la columna fare_amount es: double


In [71]:
#Ultimas 3 filas del dataset
df.tail(3)

[Row(vendor_id='CMT', pickup_datetime='2010-01-09 14:00:44', dropoff_datetime='2010-01-09 14:14:33', passenger_count=1, trip_distance=0.9, pickup_longitude=-74.00217, pickup_latitude=40.721619, rate_code='1', store_and_fwd_flag='0', dropoff_longitude=-73.988528, dropoff_latitude=40.721484, payment_type='Cas', fare_amount=8.099999999999998, surcharge=0.0, mta_tax=0.5, tip_amount=0.0, tolls_amount=0.0, total_amount=8.599999999999998),
 Row(vendor_id='CMT', pickup_datetime='2010-01-09 09:52:23', dropoff_datetime='2010-01-09 09:59:41', passenger_count=1, trip_distance=2.0, pickup_longitude=-73.98006599999998, pickup_latitude=40.770469, rate_code='1', store_and_fwd_flag='0', dropoff_longitude=-73.95807499999998, dropoff_latitude=40.783242, payment_type='Cas', fare_amount=7.3, surcharge=0.0, mta_tax=0.5, tip_amount=0.0, tolls_amount=0.0, total_amount=7.8),
 Row(vendor_id='CMT', pickup_datetime='2010-01-05 15:25:59', dropoff_datetime='2010-01-05 15:33:54', passenger_count=1, trip_distance=1.0

In [72]:
#Nulos en passenger_count
nulos_p = df.where(df["passenger_count"].isNull()).count()
print(f'La columna Passenger_count tiene {nulos_p} datos nulos')

La columna Passenger_count tiene 0 datos nulos
